# Tree Models

## Objectives
The objective of this notebook is to fit tree-based models to the house prices and compare them with the linear models. The linear models left heavy tails, a larger spread for cheap houses and large extrapolations, which suggests structure that trees can represent. Sections 1 to 3 explain a single decision tree, bagging and boosting.

## 1. Decision Tree
A regression tree asks yes or no questions about one column at a time, and predicts the mean price of the train houses in the leaf a house ends up in. The columns and values below are made up to show the shape.

```mermaid
flowchart TD
    A["OverallQual <= 6?"] -->|Yes| B["GrLivArea <= 1500?"]
    A -->|No| C["GrLivArea <= 2000?"]
    B -->|Yes| D["Leaf: mean price"]
    B -->|No| E["Leaf: mean price"]
    C -->|Yes| F["Leaf: mean price"]
    C -->|No| G["Leaf: mean price"]
```

- The root is the top node, the branches are the nodes with a question, and the leaves are the nodes at the bottom. A yes goes left.
- A classification tree predicts the most common category in the leaf, and splits by how pure the categories are.

### Choosing a Split
At each node we try every column $j$ and every threshold $s$, and keep the pair that minimises the squared error of the two groups, each predicted by its mean:
$$\min_{j,\, s} \left[ \sum_{i \in R_L} (y_i - \bar y_L)^2 + \sum_{i \in R_R} (y_i - \bar y_R)^2 \right], \qquad R_L = \{x_j \le s\},\; R_R = \{x_j > s\}$$
The thresholds are the midpoints between the sorted values of the column. The search is greedy: it picks the best split for this node, then repeats in each group.

### Pruning
A tree grown until every leaf has one house fits the noise: a low bias and a high variance. There are two ways to control it:
- Pre-pruning stops the growth early, with a limit on the depth, on the houses per leaf, or on how much a split must reduce the error. We choose the limit by cross-validation.
- Post-pruning grows a large tree and cuts it back by minimising $\text{RSS}(T) + \alpha |T|$, where $|T|$ is the number of leaves. Raising $\alpha$ removes the branches that reduce the error least per leaf, one at a time, which gives a sequence of smaller trees. We choose $\alpha$ by cross-validation. This $\alpha$ is not the penalty of the regularisation notebook.

### Properties
- No standardising or log of the columns is needed, because a split only uses the order of the values.
- Interactions are found by themselves.
- Predictions do not extrapolate, because a house outside the train range falls in the outermost leaf.
- The variance is high, because a small change in the houses can change an early split.

## 2. Bagging
Bagging, short for bootstrap aggregating, reduces the variance of any model: fit the model to many bootstrap samples and average the predictions. It helps most for high-variance models, so it is usually applied to deep trees. The steps are:
- Draw a bootstrap sample: $n$ houses drawn with replacement from the $n$ train houses, so some appear twice and some not at all.
- Fit a model to it, here a deep tree.
- Repeat for $B$ samples. A prediction is the average of the models, or the majority vote for a classification.

### Random Forest
A random forest is bagged trees with one extra step: each split tries only a random subset of the columns.

```mermaid
flowchart LR
    D["Train houses"] --> S1["Bootstrap sample 1"]
    D --> S2["Bootstrap sample 2"]
    D --> S3["Bootstrap sample B"]
    S1 --> T1["Deep tree 1"]
    S2 --> T2["Deep tree 2"]
    S3 --> T3["Deep tree B"]
    T1 --> M["Average"]
    T2 --> M
    T3 --> M
```

If the $B$ predictions each have variance $\sigma^2$ and a correlation $\rho$ between any two, the variance of their average is
$$\rho\,\sigma^2 + \frac{1 - \rho}{B}\,\sigma^2$$
- The second term goes to 0 as $B$ grows, so more trees never overfit.
- The first term does not, so the trees must differ from each other. The bootstrap samples make them differ. The random columns of a forest make them differ more, as otherwise the strongest columns would be the first split of nearly every tree.
- Averaging leaves the bias about the same, so the trees are grown deep.

### Out-of-Bag Error
A house is missing from a bootstrap sample with probability $(1 - 1/n)^n \approx 0.368$, so each tree misses about a third of the houses. Predicting each house with only the trees that missed it, and measuring the error, gives the out-of-bag error: an estimate for unseen houses with no separate validation set. We use it to choose the number of columns tried at each split. A common start is a third of the columns for regression.

## 3. Boosting
Boosting reduces the bias instead of the variance. It starts from a simple model, and repeatedly fits a small tree to the mistakes the model still makes, then adds a fraction of that tree to the model. For squared error the steps are:
- Start with the mean of the train prices as the prediction for every house.
- Compute the residuals, the amount by which the current model is wrong for each house.
- Fit a small tree to the residuals, so each leaf predicts the mean residual of its houses.
- Add the tree to the model after multiplying it by a learning rate $\eta$ between 0 and 1: $F_m = F_{m-1} + \eta \times \text{tree}$. Then repeat from the residuals.

The residual is the negative gradient of the squared error, so each step is a gradient descent step in the space of functions, which is why it is called gradient boosting.

For example, take four houses with targets 2, 4, 6 and 12, one-split trees and $\eta = 0.5$. The numbers are made up.
- The mean predicts 6 for all, the residuals are -4, -2, 0 and 6, and the squared error is 56.
- Tree 1 separates the first three houses from the last, with leaf values -2 and 6. The predictions become 5, 5, 5 and 9, and the error falls to 20.
- Tree 2 fits the new residuals, -3, -1, 1 and 3, with leaf values -2 and 2 for the first two houses and the last two. The predictions become 4, 4, 6 and 10, and the error falls to 8.

Each tree is too simple to fit the houses alone, but their sum can. In practice there are 100 or more trees of 3 to 6 levels. A smaller learning rate generalises better but needs more trees. Unlike a forest, boosting can overfit if it runs too long, so the number of trees is chosen by validation error. XGBoost and LightGBM are fast implementations of this idea, with extra regularisation of the trees.